In [ ]:
import pandas as pd

df = pd.read_csv("chats_sample_baseline.csv")

# print(df.head())
# print(df.columns)
# print(df['label'].value_counts())
print(df.isnull().sum())

In [ ]:
df = df[df['label'] != -1]
print(df['label'].value_counts())
print(df.isnull().sum())

## Baseline Models

A baseline model is a simple model that provides a reference point for evaluating more complex models. It's often the simplest possible solution to a problem and helps to establish a minimum performance benchmark. If a more sophisticated model cannot outperform the baseline, it indicates that the complexity might not be justified or that there are issues with the more advanced model's implementation or data.

In this context, predicting the `label` based on the `message` content can be considered a baseline classification problem. A very simple baseline could be to predict the majority class for all messages, or to use a simple text classification model (e.g., Naive Bayes or a simple Logistic Regression) without extensive feature engineering or complex neural network architectures. This initial model will give us a performance metric to compare against more advanced models we might develop later.

# Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = df['message']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

### Understanding `random_state`

The `random_state` parameter, commonly found in functions that involve random operations (like `train_test_split`, `KFold`, `RandomForestClassifier`, etc.), is crucial for **reproducibility**.

*   **Pseudorandomness**: Many algorithms that appear to be random are actually *pseudorandom*. This means they use a deterministic process to generate a sequence of numbers that appear random, but are entirely predictable if you know the starting point.

*   **The Seed**: The integer value passed to `random_state` acts as a 'seed' for this pseudorandom number generator. If you use the same seed value, the generator will produce the exact same sequence of 'random' numbers every time you run the code.

*   **Ensuring Consistency**: By setting `random_state` to a specific integer (e.g., `42`, `0`, or any other integer), you ensure that:
    *   Your data splits (like `X_train`, `X_test`) are always identical across multiple runs.
    *   Any algorithms that use internal randomness (e.g., initializing weights in neural networks, selecting features in tree-based models) will behave consistently.

*   **No `random_state`**: If `random_state` is not set, or set to `None`, the random number generator will typically use the current system time as a seed. This means that each time you run your code, you'll get a different 'random' result, making it difficult to debug, compare models, or share reproducible findings.

### Understanding `stratify`

The `stratify` parameter in `train_test_split` is used to ensure that the proportions of class labels are preserved in both the training and testing sets. This is particularly important when dealing with imbalanced datasets, where some classes have significantly fewer samples than others.

*   **Class Proportions**: When `stratify=y` is set, `train_test_split` will distribute the samples such that the percentage of each class in the `y_train` and `y_test` subsets is approximately the same as the percentage of each class in the original `y` dataset.

*   **Why it's Important**: Without stratification, a random split could, by chance, assign all or most of the samples from a minority class to either the training or testing set. This can lead to:
    *   **Poor model training**: If the training set lacks sufficient examples of a minority class.
    *   **Unreliable evaluation**: If the testing set doesn't accurately represent the class distribution of the real-world data.

*   **Example**: If your original dataset has 90% class 0 and 10% class 1, then with `stratify=y`, both your training set and your test set will also contain approximately 90% class 0 and 10% class 1. This provides a more robust and realistic evaluation of your model's performance across all classes.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer( stop_words='english', max_features=3000 )
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

X_train_tfidf.shape, X_test_tfidf.shape

### Understanding TF-IDF Vectorization

**TF-IDF** (Term Frequency-Inverse Document Frequency) is a numerical statistic that reflects how important a word is to a document in a collection or corpus. It's often used as a weighting factor in information retrieval and text mining.

*   **Term Frequency (TF)**: Measures how frequently a term appears in a document. The more often a word appears in a document, the higher its TF.
*   **Inverse Document Frequency (IDF)**: Measures how important a term is across the whole corpus. If a word appears in many documents, its IDF value will be low (meaning it's not very discriminating). If a word appears in only a few documents, its IDF will be high (meaning it's more specific and potentially more important).

When TF and IDF are multiplied, words that are frequent in a document but rare across the corpus get a higher TF-IDF score, making them more significant.

### `TfidfVectorizer` Parameters

The `TfidfVectorizer` from `sklearn.feature_extraction.text` converts a collection of raw documents to a matrix of TF-IDF features. Here are the parameters used:

*   **`stop_words='english'`**: This parameter tells the vectorizer to remove common English stop words (like "the", "is", "a") from the text. These words are usually very frequent but carry little meaning for text classification, so removing them helps to reduce noise and the dimensionality of the feature space.
*   **`max_features=3000`**: This parameter limits the number of features (i.e., unique words or n-grams) to consider. The vectorizer will select the top 3000 features ordered by term frequency across the corpus. This helps to manage the dimensionality of the feature matrix, especially with large text datasets, preventing the 'curse of dimensionality' and improving computational efficiency.

### Why `X_test` is only `transform`ed and not `fit_transform`ed

This is a critical concept in machine learning, particularly when dealing with data preprocessing:

*   **`fit()`**: This method learns the vocabulary and IDF weights from the input data. In the context of `TfidfVectorizer`, `fit()` analyzes the training text (`X_train`) to identify all unique words (vocabulary) and calculates their inverse document frequencies.
*   **`transform()`**: This method uses the learned vocabulary and IDF weights to convert the text into a TF-IDF matrix.
*   **`fit_transform()`**: This method combines both `fit()` and `transform()` in a single step.

We apply `fit_transform()` only to the **training data (`X_train`)** (`vectorizer.fit_transform(X_train)`). This means the vectorizer learns its vocabulary and IDF weights *only* from the training set. This is essential to prevent **data leakage**.

**Data leakage** occurs when information from the test set is inadvertently used during the training phase. If we were to `fit_transform()` the test set (`X_test`), the vectorizer would learn new words or adjust its IDF weights based on the test set's content. This would give our model an unfair advantage, as it would have 'seen' characteristics of the test data during its feature engineering, leading to an overly optimistic evaluation of its performance on unseen data.

Therefore, to simulate a real-world scenario where the model encounters entirely new, unseen data, we only `transform()` the test data (`X_test_tfidf = vectorizer.transform(X_test)`). The `transform()` method uses the vocabulary and IDF weights *already learned* from the training data to convert the test set's text into TF-IDF features, ensuring that the test set remains truly independent for evaluation.

### Why TF-IDF over other Vectorizers?

Choosing TF-IDF for this baseline model, especially with a relatively small dataset and for classification, offers several advantages over more complex vectorization methods like those based on BERT or other deep learning embeddings:

*   **Simplicity and Interpretability**: TF-IDF is a straightforward and highly interpretable method. Its scores directly reflect the importance of words in documents, making it easier to understand why certain features might contribute to a classification decision. This simplicity is ideal for a baseline, as it provides a clear reference point.

*   **Small Sample Size**: For datasets with a limited number of samples, like the one we are working with (429 messages), complex deep learning models (e.g., BERT, Word2Vec, GloVe) often struggle. These models typically require vast amounts of data to learn robust and generalized representations. With insufficient data, they are prone to learning noise and performing poorly.

*   **Risk of Overfitting with Complex Embeddings**: While powerful, word embeddings from models like BERT capture highly nuanced semantic relationships and generate very high-dimensional feature vectors. When applied to a small dataset, these rich representations can easily lead to **overfitting**. An overfit model performs exceptionally well on the training data but fails to generalize to unseen test data, producing unreliable evaluations. TF-IDF, by contrast, creates a sparser and lower-dimensional representation (especially with `max_features` set), which is less likely to overfit small datasets.

*   **Computational Efficiency**: TF-IDF is computationally less intensive to train and apply compared to generating embeddings from large pre-trained language models, which can be significant for quick experimentation and baseline establishment.

# Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42, # For reproducibility
    solver='liblinear' # Good for small datasets and 'balanced' class_weight
)

model.fit(X_train_tfidf, y_train)

### Understanding Logistic Regression Parameters

Let's break down the key parameters used in the `LogisticRegression` model:

*   **`max_iter=1000`**: This parameter sets the maximum number of iterations for the optimization algorithm to converge. Logistic Regression uses an iterative process to find the optimal coefficients that best fit the data. If the model doesn't reach convergence (a stable solution) within the default number of iterations (often 100), increasing `max_iter` gives it more attempts. This is particularly useful for larger or more complex datasets where convergence might take longer.

*   **`class_weight='balanced'`**: This is crucial for **imbalanced datasets**, where one class has significantly more samples than another (as is the case with your `label` data). When set to `'balanced'`, the algorithm automatically adjusts the weights inversely proportional to the class frequencies. This means it assigns a higher weight to samples from the minority class and a lower weight to samples from the majority class during training. This prevents the model from being biased towards the majority class and helps it learn to classify the minority class more effectively.

*   **`random_state=42`**: This parameter ensures **reproducibility** for any random aspects of the model's training process. Many machine learning algorithms, including some logistic regression solvers, involve random initialization or shuffling. By setting `random_state` to a specific integer (like `42`), you fix the seed for the internal random number generator. This guarantees that if you run your code multiple times, you will get the exact same results, making your experiments consistent and debuggable.

*   **`solver='liblinear'`**: This parameter specifies the algorithm used to optimize the model. Different solvers are suited for different types of problems and datasets:
    *   `'liblinear'` is a good choice for smaller datasets and is efficient for both L1 and L2 regularization. It is generally recommended when `class_weight='balanced'` is used.
    *   Other common solvers include `'lbfgs'`, `'newton-cg'`, `'sag'`, and `'saga'`, each with their own strengths and use cases. Choosing the right solver can impact training speed and model performance.

In [ ]:
y_pred = model.predict(X_test_tfidf)

In [ ]:
y_prob = model.predict_proba(X_test_tfidf)[:, 1]

# Evaluation

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))

### Understanding Accuracy

**Accuracy** represents the proportion of correctly classified instances (both true positives and true negatives) out of the total number of instances.

**How it's Measured:**

It is calculated using the following formula:

$$ \text{Accuracy} = \frac{\text{Number of Correct Predictions}}{\text{Total Number of Predictions}} = \frac{\text{TP} + \text{TN}}{\text{TP} + \text{TN} + \text{FP} + \text{FN}} $$

Where:
*   **TP (True Positives)**: Instances correctly predicted as positive.
*   **TN (True Negatives)**: Instances correctly predicted as negative.
*   **FP (False Positives)**: Instances incorrectly predicted as positive (Type I error).
*   **FN (False Negatives)**: Instances incorrectly predicted as negative (Type II error).

**Interpretation:**
An accuracy score of 0.895 (or 89.5%) means that the model correctly predicted the label for approximately 89.5% of the messages in the test set.

While accuracy is easy to understand, it can be misleading in cases of highly imbalanced datasets, where a high accuracy might be achieved by simply predicting the majority class most of the time. In such scenarios, other metrics like precision, recall, and F1-score (which are shown in the classification report) provide a more comprehensive view of model performance.

In [ ]:
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

### Understanding Precision, Recall, and F1-Score (from Classification Report)

While accuracy provides a general overview, for imbalanced datasets (like ours, where one class is much rarer than the other), it can be misleading. Precision, Recall, and F1-score offer a more nuanced understanding of model performance, especially concerning the minority class.

Let's define the terms we'll use:
*   **True Positives (TP)**
*   **True Negatives (TN)**
*   **False Positives (FP)** - Type I error
*   **False Negatives (FN)** - Type II error

Here's what each metric means:

*   **Precision**:
    *   **Formula**: $$ \text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}} $$
    *   **Interpretation**: Precision answers the question: "Of all instances the model predicted as positive, how many were actually positive?" It measures the accuracy of positive predictions. A high precision indicates a low false positive rate.
    *   **Example**: For class `1` (positive class), a precision of 0.40 means that when the model predicts a message is `1`, it is correct only 40% of the time. The other 60% of the time, it incorrectly classified a `0` as a `1` (false positive).

*   **Recall (Sensitivity)**:
    *   **Formula**: $$ \text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}} $$
    *   **Interpretation**: Recall answers the question: "Of all the actual positive instances, how many did the model correctly identify?" It measures the model's ability to find all positive samples. A high recall indicates a low false negative rate.
    *   **Example**: For class `1`, a recall of 0.57 means that the model was able to correctly identify 57% of all actual `1` messages. The remaining 43% of actual `1` messages were missed by the model (false negatives).

*   **F1-Score**:
    *   **Formula**: $$ \text{F1-Score} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}} $$
    *   **Interpretation**: The F1-score is the harmonic mean of precision and recall. It tries to find a balance between the two metrics. A high F1-score means that the model has good precision and good recall. It is particularly useful when you need a balance between precision and recall, especially in cases of imbalanced classes.
    *   **Example**: For class `1`, an F1-score of 0.47 suggests a moderate balance between identifying actual `1` messages and avoiding false alarms. Given the low precision, the F1-score is pulled down by it.

*   **Support**:
    *   **Interpretation**: The number of actual occurrences of each class in the specified dataset (`y_test` in this case).
    *   **Example**: For class `0`, the support is 79, and for class `1`, it is 7. This highlights the class imbalance.

*   **Macro Avg**: The average of the metrics (precision, recall, F1-score) calculated independently for each class. It treats all classes equally, regardless of their support.

*   **Weighted Avg**: The average of the metrics, where each class's score is weighted by its support (the number of true instances for each class). This is useful when you want to consider the contribution of each class based on its frequency in the dataset.

**Conclusion from our report:**

For class `0` (the majority class), the model performs very well across all metrics (high precision, recall, and F1-score), which is expected due to its dominance. However, for class `1` (the minority class), the precision (0.40) and recall (0.57) are significantly lower, resulting in a moderate F1-score (0.47). This indicates that while the model can identify some of the positive instances, it also produces a considerable number of false positives and misses a notable portion of actual positive instances. This is a common challenge with imbalanced datasets and suggests that further efforts might be needed to improve the model's performance on the minority class.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

print(cm)
#[[TN FP]
# [FN TP]]

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Non-Urgent', 'Urgent'],
    yticklabels=['Non-Urgent', 'Urgent']
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc

fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 4))

plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.2f}')
plt.plot([0, 1], [0, 1], linestyle='--')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()

plt.show()

### Understanding ROC Curve and AUC

The **Receiver Operating Characteristic (ROC) curve** is a graphical plot that illustrates the diagnostic ability of a binary classifier system as its discrimination threshold is varied. It plots two parameters:

*   **True Positive Rate (TPR)**: Also known as Recall or Sensitivity, it's the proportion of actual target cases that are correctly identified as target. $$ \text{TPR} = \frac{\text{TP}}{\text{TP} + \text{FN}} $$
*   **False Positive Rate (FPR)**: The proportion of actual non-target cases that are incorrectly identified as target. $$ \text{FPR} = \frac{\text{FP}}{\text{FP} + \text{TN}} $$

**Interpretation of the ROC Curve:**
*   The curve is plotted with TPR on the y-axis and FPR on the x-axis.
*   A model that perfectly classifies (100% TPR, 0% FPR) would have its curve pass through the top-left corner (0,1) of the plot.
*   A purely random classifier would produce a diagonal line from (0,0) to (1,1). This is often called the "line of no-discrimination" or the "chance line."
*   Models performing better than random will have their curve above this diagonal line.

**Area Under the Curve (AUC):**

*   **AUC** (Area Under the ROC Curve) is a single scalar value that summarizes the overall performance of a classifier across all possible classification thresholds.
*   It represents the probability that the classifier will rank a randomly chosen target instance higher than a randomly chosen non-target instance.
*   **Interpretation of AUC:**
    *   An AUC of 1.0 indicates a perfect classifier.
    *   An AUC of 0.5 indicates a classifier no better than random guessing.
    *   An AUC less than 0.5 indicates a classifier that performs worse than random (e.g., consistently predicts the opposite of the actual class), which usually means there's an issue with the model or data.

In our plot, the AUC of 0.76 suggests that our model has a fair ability to distinguish between target and non-target classes, performing moderately better than random. Given the class imbalance, this is a more robust metric than accuracy for assessing the model's predictive power for both classes.

In [ ]:
from sklearn.metrics import precision_recall_curve, auc

precision, recall, thresholds = precision_recall_curve(y_test, y_prob)
auprc = auc(recall, precision)

plt.figure(figsize=(6, 4))

plt.plot(recall, precision, label=f'AUPRC = {auprc:.2f}')

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()

plt.show()

### Understanding the Precision-Recall Curve

The **Precision-Recall (PR) curve** is another valuable tool for evaluating the performance of binary classifiers, especially when dealing with **imbalanced datasets** (where one class is significantly under-represented compared to the other). While the ROC curve considers both True Positives (TP) and True Negatives (TN), the PR curve focuses primarily on the positive class (our 'target' class).

**Components of the PR Curve:**

*   **Precision (Y-axis)**: $$ \text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}} $$
    Measures the proportion of true positive predictions among all positive predictions made by the model. It answers: "Of all that the model predicted as target, how many were actually target?"

*   **Recall (X-axis)**: $$ \text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}} $$
    Measures the proportion of true positive predictions among all actual positive instances. It answers: "Of all the actual target instances, how many did the model correctly identify?"

**Why is it useful for Imbalanced Datasets?**

In imbalanced datasets, the number of True Negatives (TN) is often very high. The ROC curve's False Positive Rate (FPR) uses TN in its denominator (FPR = FP / (FP + TN)). If TN is large, FPR can remain low even with many False Positives (FP), making the ROC curve look overly optimistic.

The Precision-Recall curve, however, does not involve True Negatives. It directly highlights the trade-off between identifying true positive cases (Recall) and making accurate positive predictions (Precision). When you have a high number of False Positives or False Negatives, the PR curve will show this more clearly than an ROC curve.

**Interpretation of the PR Curve:**

*   A perfect classifier would have a curve that goes straight up to (0, 1) and then across to (1, 1). This indicates 100% precision for all recall values.
*   The curve generally starts at a high precision (low recall) and drops as recall increases. This is because to achieve higher recall (find more actual positives), the model often has to make more positive predictions, potentially including more false positives, thus reducing precision.
*   A model that performs well on an imbalanced dataset will have a curve that stays as close to the top-right corner as possible.

**Area Under the Precision-Recall Curve (AUPRC or PR AUC):**

Similar to AUC for ROC, the area under the PR curve can be calculated. A higher AUPRC indicates better performance. It provides a single number summary of the model's performance concerning the positive class, balancing precision and recall. A high AUPRC means the model has both high precision and high recall across different thresholds.

Our Precision-Recall curve shows a typical trade-off, where as we aim for higher recall (identifying more 'Urgent' messages), the precision (accuracy of those 'Urgent' predictions) tends to decrease. The AUPRC for our model is 0.47, which suggests a moderate performance in balancing precision and recall for the positive class. This again emphasizes the challenge of classifying the minority class in our imbalanced dataset.

# Failure Case Analysis

Analyzing False Negatives

In [ ]:
results_df = X_test.copy()

results_df = pd.DataFrame({
    'message': X_test,
    'actual': y_test,
    'predicted': y_pred,
    'probability': y_prob
})
results_df

In [ ]:
false_negatives = results_df[
    (results_df['actual'] == 1) &
    (results_df['predicted'] == 0)
]

false_negatives

In [ ]:
false_positives = results_df[
    (results_df['actual'] == 0) &
    (results_df['predicted'] == 1)
]

false_positives

In [ ]:
pd.set_option('display.max_colwidth', None)
false_positives['message']